# Stage 1 — The policy environment

One worked reform, top to bottom: raising the rate at which earnings between 520 € and
1000 € per month are withdrawn from Bürgergeld, so the retained share on that band falls
from 30 % to 15 %.

We work on 1 July 2023, the date on which the Bürgergeld-Gesetz's new Freibetrag
schedule took effect. The same policy date runs through all three notebooks.

In [ ]:
import numpy as np
import pandas as pd

from gettsim import (
    InputData,
    MainTarget,
    TTTargets,
    copy_environment,
    main,
)

POLICY_DATE = "2023-07-01"

## 1 · The policy environment

`main` returns the nested dictionary of every parameter and function that holds on a
given date.

In [ ]:
status_quo = main(
    main_target=MainTarget.policy_environment,
    policy_date_str=POLICY_DATE,
)

sorted(status_quo)

`bürgergeld` is the namespace this notebook works in.

In [ ]:
sorted(status_quo["bürgergeld"])

The leaf we are after is the Freibetrag schedule: how much earned income stays with the
household.

In [ ]:
freibetrag = status_quo["bürgergeld"][
    "parameter_anrechnungsfreies_einkommen_ohne_kinder_in_bg"
]
type(freibetrag)

In [ ]:
freibetrag.value

Between 100 € and 520 € of monthly earnings, 20 % of each additional euro stays with the
household; between 520 € and 1000 €, 30 % does. That middle band is the one we change.

## 2 · Running GETTSIM once

`gettsim-personas` provides ready-made example households; we take a couple with one
child on basic income support and compute the status-quo Bürgergeld.

In [ ]:
from gettsim_personas import grundsicherung_für_erwerbsfähige

persona = grundsicherung_für_erwerbsfähige.Couple1Child(policy_date_str=POLICY_DATE)
print(persona.description)

In [ ]:
TARGETS = {"bürgergeld": {"betrag_m_bg": None}}

baseline = main(
    main_target=MainTarget.results.df_with_nested_columns,
    policy_date=persona.policy_date,
    input_data=InputData.tree(persona.input_data_tree),
    tt_targets=TTTargets.tree(TARGETS),
    include_warn_nodes=False,
)
baseline

### Which inputs does a target need?

Ask GETTSIM rather than guessing.

In [ ]:
template = main(
    main_target=MainTarget.templates.input_data_dtypes.tree,
    policy_date_str=POLICY_DATE,
    tt_targets=TTTargets.tree(TARGETS),
    include_warn_nodes=False,
)


def qnames(tree, prefix=""):
    for key, value in tree.items():
        path = f"{prefix}__{key}" if prefix else key
        if isinstance(value, dict):
            yield from qnames(value, path)
        else:
            yield path


print(f"{len(list(qnames(template)))} inputs required")
list(qnames(template))[:10]

## 3 · Changing a parameter

Three steps: copy the environment, build the new parameter object, put it back.

In [ ]:
reform = copy_environment(status_quo)

`get_piecewise_parameters` rebuilds a piecewise schedule from a list of intervals. On the
band between 520 € and 1000 € the retained share goes from 30 % to 15 %, a withdrawal
rate of 85 % instead of 70 %.

In [ ]:
from gettsim.tt import (
    PiecewisePolynomialParam,
    TTSIMUnit,
    get_piecewise_parameters,
)

steeper = PiecewisePolynomialParam(
    value=get_piecewise_parameters(
        func_type="piecewise_linear",
        parameter_list=[
            {"interval": "(-inf, 0)", "intercept": 0, "slope": 0},
            {"interval": "[0, 100)", "slope": 1.0},
            {"interval": "[100, 520)", "slope": 0.2},
            {"interval": "[520, 1000)", "slope": 0.15},  # was 0.3
            {"interval": "[1000, 1200)", "slope": 0.1},
            {"interval": "[1200, inf)", "slope": 0.0},
        ],
        leaf_name="parameter_anrechnungsfreies_einkommen_ohne_kinder_in_bg",
        xnp=np,
    ),
    input_unit=TTSIMUnit.EUR.PER_MONTH,
    output_unit=TTSIMUnit.EUR.PER_MONTH,
)

reform["bürgergeld"]["parameter_anrechnungsfreies_einkommen_ohne_kinder_in_bg"] = (
    steeper
)

Hand the modified environment to `main` as `policy_environment=` and compare.

In [ ]:
after = main(
    main_target=MainTarget.results.df_with_nested_columns,
    policy_date=persona.policy_date,
    policy_environment=reform,
    input_data=InputData.tree(persona.input_data_tree),
    tt_targets=TTTargets.tree(TARGETS),
    include_warn_nodes=False,
)

pd.DataFrame(
    {
        "status quo": baseline.to_numpy().ravel(),
        "reform": after.to_numpy().ravel(),
    }
)

Other parameter classes are built differently; `gettsim/docs/how_to_guides/modifications_of_policy_environments.ipynb`
has a worked example of each.

## 4 · Changing a function

GETTSIM picks between two Freibetrag schedules depending on whether children live in the
Bedarfsgemeinschaft. Making the more generous one apply only from the second child needs
a new function and a new parameter. First read the function we are about to replace.

In [ ]:
import inspect

print(
    inspect.getsource(
        status_quo["bürgergeld"]["anrechnungsfreies_einkommen_m"].function
    )
)

The replacement carries the same name, signature and date range. Arguments are other
nodes, addressed by qname — a double underscore separates namespaces.

In [ ]:
from types import ModuleType

from gettsim.tt import (
    PiecewisePolynomialParamValue,
    ScalarParam,
    piecewise_polynomial,
    policy_function,
)


@policy_function(
    start_date="2023-01-01",
    unit=TTSIMUnit.CURRENCY.PER_MONTH,
)
def anrechnungsfreies_einkommen_m(
    einnahmen__bruttolohn_m: float,
    einkommensteuer__einkünfte__aus_selbstständiger_arbeit__betrag_m: float,
    familie__anzahl_kinder_bis_17_bg: int,
    parameter_anrechnungsfreies_einkommen_ohne_kinder_in_bg: PiecewisePolynomialParamValue,
    parameter_anrechnungsfreies_einkommen_mit_kindern_in_bg: PiecewisePolynomialParamValue,
    min_anzahl_kinder_für_höheren_freibetrag: int,
    xnp: ModuleType,
) -> float:
    """Use the with-children schedule only from the nth child onwards."""
    erwerbseinkommen_m = (
        einnahmen__bruttolohn_m
        + einkommensteuer__einkünfte__aus_selbstständiger_arbeit__betrag_m
    )
    # Call piecewise_polynomial inside each branch rather than selecting the parameter
    # object first: GETTSIM vectorizes these functions, and a branch that returns a
    # parameter object would be vectorized into an array of them.
    if familie__anzahl_kinder_bis_17_bg >= min_anzahl_kinder_für_höheren_freibetrag:
        out = piecewise_polynomial(
            x=erwerbseinkommen_m,
            parameters=parameter_anrechnungsfreies_einkommen_mit_kindern_in_bg,
            xnp=xnp,
        )
    else:
        out = piecewise_polynomial(
            x=erwerbseinkommen_m,
            parameters=parameter_anrechnungsfreies_einkommen_ohne_kinder_in_bg,
            xnp=xnp,
        )
    return out


reform_zweites_kind = copy_environment(status_quo)
reform_zweites_kind["bürgergeld"]["anrechnungsfreies_einkommen_m"] = (
    anrechnungsfreies_einkommen_m
)
reform_zweites_kind["bürgergeld"]["min_anzahl_kinder_für_höheren_freibetrag"] = (
    ScalarParam(value=2, unit=TTSIMUnit.COUNT.PER_BG)
)

Run that environment on the same persona.

In [ ]:
main(
    main_target=MainTarget.results.df_with_nested_columns,
    policy_date=persona.policy_date,
    policy_environment=reform_zweites_kind,
    input_data=InputData.tree(persona.input_data_tree),
    tt_targets=TTTargets.tree(TARGETS),
    include_warn_nodes=False,
)

---

Keep `status_quo` and `reform` around: notebook 02 picks the reform environment up
tomorrow.